# Experiment 01 · CartPole PPO

**WalkingLab × Hands-On Modern RL companion experiment notebook**

Train PPO on CartPole-v1 with CPU, inspect evaluation rewards, and render the learned policy.

- Resource profile: **CPU**
- Quick run in this notebook: **30,000** training units
- Full experiment: 30,000 environment steps (usually under one minute on a notebook CPU)
- [Live ModelScope Studio](https://modelscope.cn/studios/walkinglab/hands-on-modern-rl-experiment01-cartpole)
- [Experiment source](https://github.com/walkinglabs/hands-on-modern-rl/tree/main/modelscope-space/hands-on-modern-rl-experiment01-cartpole)
- [Hands-On Modern RL](https://github.com/walkinglabs/hands-on-modern-rl) · [WalkingLab](https://modelscope.cn/organization/walkinglab)

The notebook imports the exact runtime used by the Studio. Change the parameters below, run the cells in order,
and compare the checkpoint curve with the final policy GIF or result image. The first setup can take longer because
native environments and simulator assets are cached; later runs reuse `/mnt/workspace/hands-on-modern-rl-notebooks`.


## 1. Question and run boundary

This experiment asks whether the selected policy improves on the task's evaluation metric as its training budget
increases. Start with the quick budget to verify the environment and logs. Then increase the budget only after the
complete result cell produces a curve and an artifact.

A normal ModelScope **CPU Notebook** is sufficient; no GPU is required.

A short smoke run proves that the pipeline executes; it does not prove convergence. Use the full budget above when
comparing algorithms or reporting a learned behavior.


## 2. Prepare the matching Studio runtime


In [ ]:
from __future__ import annotations

import hashlib
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/walkinglabs/hands-on-modern-rl.git"
SPACE_SLUG = "hands-on-modern-rl-experiment01-cartpole"
INSTALL_DEPENDENCIES = True

def locate_or_clone_repo() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "modelscope-space" / SPACE_SLUG).is_dir():
            return candidate
    workspace = Path("/mnt/workspace") if Path("/mnt/workspace").is_dir() else Path.cwd()
    target = workspace / "hands-on-modern-rl-notebooks" / "source"
    target.parent.mkdir(parents=True, exist_ok=True)
    if (target / ".git").is_dir():
        subprocess.run(["git", "-C", str(target), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(target)], check=True)
    return target

REPO_ROOT = locate_or_clone_repo()
SPACE_DIR = REPO_ROOT / "modelscope-space" / SPACE_SLUG
requirements = SPACE_DIR / "requirements.txt"
packages = SPACE_DIR / "packages.txt"
cache_root = Path("/mnt/workspace/hands-on-modern-rl-notebooks") if Path("/mnt/workspace").is_dir() else REPO_ROOT / ".cache" / "online-experiments"
cache_root.mkdir(parents=True, exist_ok=True)
digest = hashlib.sha256(requirements.read_bytes() + (packages.read_bytes() if packages.exists() else b"")).hexdigest()[:12]
marker = cache_root / f"{SPACE_SLUG}-{digest}.ready"

if INSTALL_DEPENDENCIES and not marker.exists():
    if packages.exists() and sys.platform.startswith("linux") and hasattr(os, "geteuid") and os.geteuid() == 0:
        system_packages = [line.strip() for line in packages.read_text().splitlines() if line.strip() and not line.startswith("#")]
        subprocess.run(["apt-get", "update"], check=True)
        subprocess.run(["apt-get", "install", "-y", "--no-install-recommends", *system_packages], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", str(requirements)], check=True)
    marker.touch()
else:
    print(f"Dependency cache ready: {marker}")

os.chdir(SPACE_DIR)
if str(SPACE_DIR) not in sys.path:
    sys.path.insert(0, str(SPACE_DIR))
print(f"Repository: {REPO_ROOT}")
print(f"Experiment runtime: {SPACE_DIR}")


## 3. Train PPO and inspect the artifacts


In [ ]:
from train import train

TIMESTEPS = 30000
OUTPUT_DIR = cache_root / "results" / SPACE_SLUG
train(total_timesteps=TIMESTEPS, output_dir=OUTPUT_DIR)


In [ ]:
from IPython.display import Image as NotebookImage, display

curve = OUTPUT_DIR / "reward-curve.png"
replay = OUTPUT_DIR / "cartpole-trained-policy.gif"
model = OUTPUT_DIR / "ppo-cartpole.zip"
for path in (curve, replay, model):
    if not path.exists():
        raise FileNotFoundError(path)
display(NotebookImage(filename=str(curve)))
display(NotebookImage(filename=str(replay)))
print("Saved model:", model)


## 4. Read the result before increasing the budget

Compare the first and last checkpoint values, then inspect the replay. A rising curve with an implausible replay can
indicate reward shaping, evaluation, or rendering problems. A flat quick run is also inconclusive: this notebook's
default budget is a pipeline check. For a training claim, rerun with **30,000 environment steps (usually under one minute on a notebook CPU)**, keep the seed fixed,
and compare at least three seeds before drawing a conclusion.
